In [1]:
import numpy as np
import sys
import os
# Get the absolute path of the main directory
main_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# Add the main directory to sys.path
sys.path.append(main_dir)
from OptFunctions.NonLinearSE import *
from OptFunctions.oneDQHarmOscillatorfunctions import*
from OptFunctions.oneDwellfunctions import *
import time

#Para usar EGNA
from EDAspy.optimization import EGNA
from EDAspy.optimization import UMDAc
from EDAspy.optimization import EMNA

import pickle

In [2]:
def _normalize(dataset:np.array, grid_size: float):
        '''
        Normalizes the solutions i.e. multiply by a constant so that the integral of the solution^2 = 1.
        Be aware that integration depends on problem definition and size!!!!
        '''
        #grid_size = L/variables     dx of integral
        #grid_size = 5/(self.n_variables-1)

        #Assuming a constant size grid we know grid size independently of problem parameters (better code-wise)
        #self.grid_size = grid_X[-1,0] - grid_X[-2, 0]

        for ind in range(len(dataset)):
                integral = 0
                for var in range(dataset.shape[1]):
                    integral = integral + (dataset[ind,:][var]**2)*grid_size ##Tener en cuenta posibles cambios en tamaño grid!!! L caja = 5 

                A = (1/integral)**0.5
                dataset[ind, :] = A * dataset[ind, :]


In [3]:
#Medidas de los experimentos se guardan en un array.shape [Nexp, 9, 5]
#porque se harán Nexp ejecuciones de los problemas
# 9 filas porque tenemos 4 estados de Oscilador, 4 de pozo y 1 de NLSE
# 5 métricas [MSE, E, E%, GP_1000, t_exc] GP_1000 queda desierto en algoritmos diferentes 
# en este orden de manera que por ejemplo los E% de los Nexp del estado Psi0 del pozo es el elemento [:, 4, 2] del array de Resultados

n_variables = [50] #Tamaños del grid (número de dimensiones del problema de optimización)
for vars in n_variables:
    Nexp = 10
    Resultados = np.empty((Nexp, 9, 5))
    ###Limit of iterations for CMAES
    print(f'\n\n########################### Experimentos con {vars} variables ###########################\n\n')
    for experimento in range(Nexp):
        print(f'\n\n########################### Ejecución de Experimentos número: {experimento+1} de {Nexp} ###########################\n\n')
        #################################################################################
        ############## Quantum Harmonic Oscillator 1D ###################################
        #################################################################################
        #First we load the optimized hyperparameters for this problem and this number of variables        
        ruta = f'EGNABO_params/{vars}Osc'
        with open(f'{ruta}.pkl', "rb") as file:
            BO_params = pickle.load(file)

        size_gen = BO_params['size_gen']
        alpha = BO_params['alpha']
        elite_factor = BO_params['elite_factor']
        whitestype = BO_params['whites']

        max_f_evals = 1*10**6 #vars*10000 #D * 10.000 so, max_f_evals = max_iter * size_gen.  
        max_iter = int(max_f_evals/size_gen)
        dead_iter = int(max_iter*0.05) #dead_iter = 5% of max_iter
        ##################################################################
        ######### MANDATORY ARCS for GBN structure #######################
        ##################################################################
        #ARCOS cadena simple
        #whites es una lista con los arcos obligatorios que aprender
        #'Continuidad de la función'
        chain=[(str(i), str(i+1)) for i in range(vars-1)]

        #ARCOS [(0, 50), (1, 49), (2, 48), ..., (23, 27), (24, 26)] conectando puntos extremos (simetría)
        # #simetría de la función SOLO PARA OSCILADOR SI ACASO
        extremes=[(str(i), str(vars-i-1)) for i in range((vars//2))]

        #ARCOS de evidencias a resto centro fuera para NLSE
        #simetría de función desde punto central (condición de contorno)
        whites1=[(str(i), str(i-1)) for i in range((vars//2),0, -1)]
        whites2=[(str(i), str(i+1)) for i in range(vars//2,vars-1, 1)]
        centertoout = whites1 + whites2
                
        #ARCOS de evidencias a resto fuera dentro para Well y Osc
        #simetría de función desde punto central (condición de contorno)
        whites1=[(str(i), str(i+1)) for i in range(0,vars//2, 1)]
        whites2=[(str(i), str(i-1)) for i in range(vars-1, (vars//2),-1)]
        outtocenter = whites1 + whites2

        ARCOS = {}
        ARCOS['chain'] = chain
        ARCOS['extremes'] = extremes
        ARCOS['centertoout'] = centertoout
        ARCOS['outtocenter'] = outtocenter
        ARCOS['None'] = None #[chain, extremes, centertoout, None]
        ##################################################################
        ##################################################################
        ##########################################################################################
        ########################### OSCILADOR ARMÓNICO ###########################################
        ##########################################################################################
        print('Quantum Harmonic Oscillator IN PROGRESS\n')
        #Problem grid definition
        X_grid = np.linspace(start=-4, stop=4, num=vars).reshape(-1,1) #para que sean n_variables filas y 1 columna
        #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
        #Los datos que conocemos son las amplitudes en x=-4 (-inf) x=4 (inf) que son 0
        X_train = X_grid[[0,-1]]
        Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array

        #Analytical solutions definitions
        #Ground state
        APsi0 = np.exp(-X_grid**2/2)
        integral =0
        for i in range(len(APsi0)):
            integral = integral + (APsi0[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi0)-1))

        B = (1/integral)**0.5
        APsi0 = B * APsi0

        #1st excitated state
        APsi1 = np.exp(-X_grid**2/2)*2*X_grid
        integral =0
        for i in range(len(APsi1)):
            integral = integral + (APsi1[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi1)-1))

        B = (1/integral)**0.5
        APsi1 = -1 * B * APsi1 #signo menos para correspondencia con solución algoritmo (simetrización)

        #2nd excitated state
        APsi2 = np.exp(-X_grid**2/2)*(4*X_grid**2-2)
        integral =0
        for i in range(len(APsi2)):
            integral = integral + (APsi2[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi2)-1))

        B = (1/integral)**0.5
        APsi2 = B * APsi2

        #3rd excitated state
        APsi3 = np.exp(-X_grid**2/2)*(8*X_grid**3-12*X_grid)
        integral =0
        for i in range(len(APsi3)):
            integral = integral + (APsi3[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi3)-1))

        B = (1/integral)**0.5
        APsi3 = -1 * B * APsi3 #signo menos para correspondencia con solución algoritmo (simetrización)

        APsis = {}
        APsis['APsi0'] = APsi0[:, 0]
        APsis['APsi1'] = APsi1[:, 0]
        APsis['APsi2'] = APsi2[:, 0]
        APsis['APsi3'] = APsi3[:, 0]
        X_grid = X_grid[:, 0]
        grid_size = X_grid[-1] - X_grid[-2]

        ##########################################################################################################
        ##########################################################################################################
        ### Ahora el código con EGNA que calcula los 4 primeros estados excitados del problema oscilador armónico
        params = {'x':X_grid}
        #Ground state
        print(f'F should be approx -0.082')
        #qegna = UMDAc(size_gen=size_gen, max_iter=800, dead_iter=201, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
        qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype])#, init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultQHOground = qegna.minimize(oneDHOExpE, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 0, 4] = end-start # 5 métricas [MSE, E, E%, GP_1000, t_exc] GP_1000 queda desierto en algoritmos diferentes 
        _normalize(dataset=egnaresultQHOground.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        #Se guardan resultados de algoritmo para siguiente estado excitado
        Psi_ground=egnaresultQHOground.sel_inds_hist[-1, 0, :-1]
        cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
        E0 = np.sqrt(-np.log(-egnaresultQHOground.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así 
        params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground}
        Resultados[experimento, 0, 1] = E0
        Resultados[experimento, 0, 2] = abs(E0-0.5)/0.5*100

        print(f'\n  |Psi0> done\n')

        #1er estado excitado
        print(f'F should be approx -4.54e-5')
        #qegna = UMDAc(size_gen=size_gen, max_iter=800, dead_iter=201, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
        qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype])#, init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultQHO1 = qegna.minimize(oneDHOExpE1, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 1, 4] = end-start
        _normalize(dataset=egnaresultQHO1.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        Psi1=egnaresultQHO1.sel_inds_hist[-1, 0, :-1]
        cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
        E1 = np.sqrt(-np.log(-egnaresultQHO1.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
        Resultados[experimento, 1, 1] = E0 + E1
        Resultados[experimento, 1, 2] = abs(E0 + E1 -1.5)/1.5*100
        
        params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1}
        print(f'\n  |Psi1> done\n')

        #2o estado excitado
        print(f'F should be approx -4.54e-5')
        #qegna = UMDAc(size_gen=size_gen, max_iter=800, dead_iter=201, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
        qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype])#, init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultQHO2 = qegna.minimize(oneDHOExpE2, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 2, 4] = end-start
        _normalize(dataset=egnaresultQHO2.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        Psi2=egnaresultQHO2.sel_inds_hist[-1, 0, :-1]
        cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
        E2 = np.sqrt(-np.log(-egnaresultQHO2.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
        Resultados[experimento, 2, 1] = E0 + E1 + E2
        Resultados[experimento, 2, 2] = abs(E0 + E1 + E2 -2.5)/2.5*100
        
        params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2 }
        print(f'\n  |Psi2> done\n')

        #3er estado excitado
        print(f'F should be approx -4.54e-5')
        #qegna = UMDAc(size_gen=size_gen, max_iter=800, dead_iter=201, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
        qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype])#, init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultQHO3 = qegna.minimize(oneDHOExpE3, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 3, 4] = end-start
        _normalize(dataset=egnaresultQHO3.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        Psi3=egnaresultQHO3.sel_inds_hist[-1, 0, :-1]
        cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
        E3 = np.sqrt(-np.log(-egnaresultQHO3.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
        Resultados[experimento, 3, 1] = E0 + E1 + E2 + E3
        Resultados[experimento, 3, 2] = abs(E0 + E1 + E2 + E3 -3.5)/3.5*100
        
        params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2, 'E3': E3, 'Psi3': Psi3 }
        print(f'\n  |Psi3> done\n')

        #Cálculos de métricas error MSE de cada estado con estado teórico
        MSEs = []
        for i in range(4):
            mse = min(np.mean((params[f'Psi{i}'] - APsis[f'APsi{i}'])**2), np.mean((-1*params[f'Psi{i}'] - APsis[f'APsi{i}'])**2))
            Resultados[experimento, i, 0] = mse # 5 métricas [MSE, E, E%, GP_1000, t_exc] GP_1000 queda desierto en algoritmos diferentes 
            MSEs.append(mse)
        print(f'MSE: {MSEs}\n')
        print('\nInfinite Oscillator DONE\n')

        ######################################################################################
        ######################################################################################
        ############## POZO DE POTENCIAL INFINITO 1D WELL ####################################
        ######################################################################################
        #First we load the optimized hyperparameters for this problem and this number of variables        
        ruta = f'EGNABO_params/{vars}Well'
        with open(f'{ruta}.pkl', "rb") as file:
            BO_params = pickle.load(file)

        size_gen = BO_params['size_gen']
        alpha = BO_params['alpha']
        elite_factor = BO_params['elite_factor']
        whitestype = BO_params['whites'] ##No es necesario redefinir ARCOS porque es igual que en caso oscilador, pero sí habrá que hacerlo en NLSE

        max_iter = int(max_f_evals/size_gen)
        dead_iter = int(max_iter*0.05) #dead_iter = 5% of max_iter
        
        print('\nInfinite Well IN PROGRESS\n')
        L=5
        #Energías teóricas
        Eg = (2*np.pi)**2/(8*L**2*1)
        En=[]
        for i in range(4):
            e=(i+1)**2*Eg
            En.append(e)

        ## Problem grid definition
        X_grid = np.linspace(start=0, stop=L, num=vars).reshape(-1,1) #para que sean n_variables filas y 1 columna
        #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
        #Los datos que conocemos son las amplitudes en x=0 x=5 que son 0
        X_train = X_grid[[0,-1]]
        Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array
        init_data = None #initialize_population(size_gen=3000, grid=X_grid, init_kernels=10, init_constraints=(X_train, Y_train))
        grid_size = X_grid[-1,0] - X_grid[-2, 0]
        params = {'L':L}

        #Soluciones teóricas
        ### Soluciones analíticas
        # def Psi_n(n):
        #     return((2/L)**0.5*np.sin(n*np.pi*np.linspace(0, L, n_variables)/L))

        APsisWell = {}
        for n in range(4):
            APsisWell[f'APsi{n}'] = (2/L)**0.5*np.sin((n+1)*np.pi*np.linspace(0, L, vars)/L)
        ###############################    
        #Ground state n=1
        ###############################
        print(f'F should be approx -0.9618')
        #qegna = UMDAc(size_gen=3000, max_iter=800, dead_iter=201, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
        qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype], init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultgroundWell = qegna.minimize(oneD_wellExpE, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 4, 4] = end-start # 5 métricas [MSE, E, E%, GP_1000, t_exc] GP_1000 queda desierto en algoritmos diferentes 
        _normalize(dataset=egnaresultgroundWell.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        Psi_fund=egnaresultgroundWell.sel_inds_hist[-1, 0, :-1]
        E_fund = np.sqrt(-np.log(-egnaresultgroundWell.sel_inds_hist[-1,0,-1])) # def exponencial de función de coste la energía se determina a partir del fitness así.
        #En este caso no hay constante multiplicando exponente exponencial
        Resultados[experimento, 4, 1] = E_fund
        Resultados[experimento, 4, 2] = abs(E_fund-En[0])/En[0]*100 

        params = {'L':L, 'E': E_fund, 'Psi': Psi_fund}
        print(f'\n  |Psi0> done\n')

        ###############################   
        #1excitated state n=2
        ###############################   
        print(f'F should be approx -0.7042')
        #qegna = UMDAc(size_gen=3000, max_iter=800, dead_iter=201, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
        qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype], init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultWellexc1 = qegna.minimize(oneD_wellExpEexcitated, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 5, 4] = end-start
        _normalize(dataset=egnaresultWellexc1.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        Psi_1=egnaresultWellexc1.sel_inds_hist[-1, 0, :-1]
        E1 = np.sqrt(-np.log(-egnaresultWellexc1.sel_inds_hist[-1,0,-1]))
        Resultados[experimento, 5, 1] = E_fund + E1
        Resultados[experimento, 5, 2] = abs(E_fund + E1 -En[1])/En[1]*100

        params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1}
        print(f'\n  |Psi1> done\n')

        ###############################   
        #2nd excitated state n=3
        ###############################   
        print(f'F should be approx -0.3775')
        #qegna = UMDAc(size_gen=3000, max_iter=800, dead_iter=201, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
        qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype], init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultWellexc2 = qegna.minimize(oneD_wellExpEexcitated3, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 6, 4] = end - start
        _normalize(dataset=egnaresultWellexc2.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        Psi_2=egnaresultWellexc2.sel_inds_hist[-1, 0, :-1]
        E2 = np.sqrt(-np.log(-egnaresultWellexc2.sel_inds_hist[-1,0,-1]))
        Resultados[experimento, 6, 1] = E_fund + E1 + E2
        Resultados[experimento, 6, 2] = abs(E_fund + E1 +E2 -En[2])/En[2]*100

        params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2}
        print(f'\n  |Psi2> done\n')

        ###############################   
        #3rd excitated state n=4
        ###############################   
        print(f'F should be approx -0.1482')
        #qegna = UMDAc(size_gen=3000, max_iter=800, dead_iter=201, n_variables=vars, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
        qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= vars, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype], init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= vars, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egnaresultWellexc3 = qegna.minimize(oneD_wellExpEexcitated4, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 7, 4] = end-start
        _normalize(dataset=egnaresultWellexc3.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
        Psi_3 = egnaresultWellexc3.sel_inds_hist[-1, 0, :-1]
        E3 = np.sqrt(-np.log(-egnaresultWellexc3.sel_inds_hist[-1,0,-1]))
        Resultados[experimento, 7, 1] = E_fund + E1 + E2 + E3
        Resultados[experimento, 7, 2] = abs(E_fund + E1 +E2+E3 -En[3])/En[3]*100

        params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2, 'E3':E3, 'Psi3': Psi_3}
        print(f'\n  |Psi3> done\n')

        #Cálculos de métricas error MSE de cada estado con estado teórico
        MSEs = []
        for i in range(4):
            mse = min(np.mean((params[f'Psi{i}'] - APsisWell[f'APsi{i}'])**2), np.mean((-1*params[f'Psi{i}'] - APsisWell[f'APsi{i}'])**2))
            Resultados[experimento, i+4, 0] = mse # 5 métricas [MSE, E, E%, GP_1000, t_exc] GP_1000 queda desierto en algoritmos diferentes 
            MSEs.append(mse)
        print(f'MSE: {MSEs}\n')  
        print('\nInfinite Well DONE\n')

        ######################################################################################
        ######################################################################################
        #################### N L S E Non linear Schroedinger Equation ########################
        ######################################################################################
        #First we load the optimized hyperparameters for this problem and this number of variables        
        ruta = f'EGNABO_params/{vars}NLSE'
        with open(f'{ruta}.pkl', "rb") as file:
            BO_params = pickle.load(file)

        size_gen = BO_params['size_gen']
        alpha = BO_params['alpha']
        elite_factor = BO_params['elite_factor']
        whitestype = BO_params['whites']
 
        max_iter = int(max_f_evals/size_gen)
        dead_iter = int(max_iter*0.05) #dead_iter = 5% of max_iter

        ##################################################################
        ######### MANDATORY ARCS for GBN structure #######################
        ##################################################################
        if vars%2 ==0:
            varsNLSE=vars+1
        else:
            varsNLSE=vars
        #ARCOS cadena simple
        #ARCOS cadena simple
        #whites es una lista con los arcos obligatorios que aprender
        #'Continuidad de la función'
        chain=[(str(i), str(i+1)) for i in range(varsNLSE-1)]

        #ARCOS [(0, 50), (1, 49), (2, 48), ..., (23, 27), (24, 26)] conectando puntos extremos (simetría)
        # #simetría de la función SOLO PARA OSCILADOR SI ACASO
        extremes=[(str(i), str(varsNLSE-i-1)) for i in range((varsNLSE//2))]

        #ARCOS de evidencias a resto centro fuera para NLSE
        #simetría de función desde punto central (condición de contorno)
        whites1=[(str(i), str(i-1)) for i in range((varsNLSE//2),0, -1)]
        whites2=[(str(i), str(i+1)) for i in range(varsNLSE//2,varsNLSE-1, 1)]
        centertoout = whites1 + whites2
                
        #ARCOS de evidencias a resto fuera dentro para Well y Osc
        #simetría de función desde punto central (condición de contorno)
        whites1=[(str(i), str(i+1)) for i in range(0,varsNLSE//2, 1)]
        whites2=[(str(i), str(i-1)) for i in range(varsNLSE-1, (varsNLSE//2),-1)]
        outtocenter = whites1 + whites2

        ARCOS = {}
        ARCOS['chain'] = chain
        ARCOS['extremes'] = extremes
        ARCOS['centertoout'] = centertoout
        ARCOS['outtocenter'] = outtocenter
        ARCOS['None'] = None #[chain, extremes, centertoout, None]
        ##################################################################
        ##################################################################
        print('\nNLSE IN PROGRESS\n')
        L=2
        X_grid = np.linspace(start=-1, stop=1, num=varsNLSE).reshape(-1,1) #para que sean n_variables filas y 1 columna
        #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
        #Los datos que conocemos son las amplitudes en x=0 que son 1
        #X_train = X_grid[n_variables//2]
        X_train = np.array((0)).reshape(-1,1)
        Y_train = np.linspace(start=1, stop=1, num=1).reshape(-1,1) #Para que sea 2D array
        
        init_data = None #initialize_population(size_gen=3000, grid=X_grid, init_kernels=10, init_constraints=(X_train, Y_train))
        params = {'L':L}
        #qegna = UMDAc(size_gen=3000, max_iter=600, dead_iter=201, n_variables=varsNLSE, lower_bound=-1, upper_bound=1, alpha=0.4)#, lower_factor=0.01)
        qegna = EGNA(size_gen=size_gen, max_iter=max_iter, dead_iter=dead_iter, n_variables= varsNLSE, lower_bound=0, upper_bound = 2, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whitestype], init_data=init_data)
        #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= varsNLSE, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
        #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= varsNLSE, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
        start = time.time()
        egparesultNLSE = qegna.minimize(NLSE, params, exp_from = 20, output_runtime=False)
        end = time.time()
        Resultados[experimento, 8, 4] = end-start # 5 métricas [MSE, E, E%, GP_1000, t_exc]
        MSE_NLSE = np.sum((1/np.cosh(X_grid[:,0]) - egparesultNLSE.sel_inds_hist[-1, 0, :-1])**2) / len(X_grid[:,0])
        MSE_NLSE = np.float64(MSE_NLSE)
        Resultados[experimento, 8, 0] = MSE_NLSE
        #setting non used columns to None
        Resultados[experimento, 8, 1:3] = None
        Resultados[experimento, :, 3] = None
        print(f'MSE: {MSE_NLSE}\n')    
        print('\nNLSE DONE\n')
        print(f'EXPERIMENT {experimento+1} of {Nexp} FINISHED\n')

    rutafinal = f'EGNA_results/{vars}'    
    with open(f'{rutafinal}.pkl', "wb") as file:
        pickle.dump(Resultados, file)



########################### Experimentos con 50 variables ###########################




########################### Ejecución de Experimentos número: 1 de 10 ###########################


Quantum Harmonic Oscillator IN PROGRESS

F should be approx -0.082
IT:  0 	Best cost:  460.1388881341059
	NFEVALS = 602500 F = -0.08268210703797836
	X = [[ 4.66316906e-05 -9.46454925e-04 -7.06821754e-04 -8.18221588e-04
  -1.96685296e-03 -3.98256650e-03 -7.54681797e-03 -1.23161950e-02
  -1.95431875e-02 -2.98350548e-02 -4.69214933e-02 -6.76161896e-02
  -9.49757003e-02 -1.29166197e-01 -1.73016624e-01 -2.25164713e-01
  -2.86704059e-01 -3.55088630e-01 -4.28098865e-01 -5.01880593e-01
  -5.73492209e-01 -6.38056165e-01 -6.91593923e-01 -7.29347319e-01
  -7.47742981e-01 -7.48553777e-01 -7.28646960e-01 -6.90050152e-01
  -6.36897727e-01 -5.71746719e-01 -5.01638898e-01 -4.28063107e-01
  -3.55677842e-01 -2.88142139e-01 -2.27269113e-01 -1.74961523e-01
  -1.30871566e-01 -9.58018013e-02 -6.77807186e-02 -4.68129066

In [3]:
#Let's check what is the preferred arc structure for the problems
import pickle
vars=50
#Quantum Harmonic Oscillator        
ruta = f'EGNABO_params/{vars}Osc'
with open(f'{ruta}.pkl', "rb") as file:
    BO_params = pickle.load(file)
print(f'The preferred arc structure for the Harmonic Oscillator is: {BO_params['whites']}\n')

#Infinite Well
ruta = f'EGNABO_params/{vars}Well'
with open(f'{ruta}.pkl', "rb") as file:
    BO_params = pickle.load(file)
print(f'The preferred arc structure for the Infinite Well is: {BO_params['whites']}\n')

#NLSE 
ruta = f'EGNABO_params/{vars}NLSE'
with open(f'{ruta}.pkl', "rb") as file:
    BO_params = pickle.load(file)    
print(f'The preferred arc structure for the NLSE problem is: {BO_params['whites']}\n')

The preferred arc structure for the Harmonic Oscillator is: chain

The preferred arc structure for the Infinite Well is: centertoout

The preferred arc structure for the NLSE problem is: centertoout

